In [45]:
import csv
import re
import datetime
from collections import defaultdict

INPUT_CSV = r"D:\утилита\gp_ddl_1_2\errors.csv"
OUT_CSV = r"D:\утилита\errors_agg.csv"

d_aggr_errors = {
    "PARTS_63" : "повторяющееся название таблицы партиции при превышение длины в 63 символа",
    "PARTS_DONT_GEN" : "не генерируются значения для списковых партиций",
    "NOT_SOURCE" : "нет источника",
    "NOT_CTE" : "нет CTE",
    "NOT_FIELD" : "отсутсвует поле",
    "SYNTAX" : "синтаксическая ошибка",
    "UNDEFINED" : "неизвестное",
    "NOT_SETTINGS_TABLE" : "нет атрибутов таблицы в сис таблице system_core.sys_object_attrs",
    "INVALID_STRUCTURE" : "невалидное формирование партиций",
    "NOT_TASK_GROUP_CODE" : "такой task_group_code нет в мете",
    "NOT_SETTINGS_ATTR" :  "нет атрибута в сис таблице system_core.sys_object_attrs"
}


def classify_error(error_text):
    '''
    возвращает список из двух элементов:
    1) название группы ошибок
    2) контекст
    '''
    if 'no partitions specified at depth 1' in error_text:
        #return ["не генерируются значения для списковых партиций", None]
        return [d_aggr_errors["PARTS_DONT_GEN"], None]
        
    if 'relation' in error_text and "already exists" in error_text:
        error_text_replace = error_text.replace('"', '')
        error_text_list = error_text_replace.split()[:4]
        if len(error_text_list[1]) == 63:
            #return ["повторяющееся название таблицы партиции при превышение длины в 63 символа", error_text_list[1]]
            return [d_aggr_errors["PARTS_63"], error_text_list[1]]

    if 'relation' in error_text and "does not exist" in error_text:
        error_text_replace = error_text.replace('""', '')
        error_text_list = error_text_replace.split()[:4]
        if '.' in error_text_list[1]:
            #return ["нет источника", error_text_list[1]]
            return [d_aggr_errors["NOT_SOURCE"], error_text_list[1]]
        else:
            #return ["нет CTE", error_text_list[1]]
            return [d_aggr_errors["NOT_CTE"], error_text_list[1]]

    if 'column' in error_text and 'does not exist' in error_text:
        error_text_list = error_text.split()
        #return ['отсутсвует поле', error_text_list[1]]
        return [d_aggr_errors["NOT_FIELD"], error_text_list[1]]

    if 'syntax error' in error_text:
        #return ['синтаксическая ошибка', None]
        return [d_aggr_errors["SYNTAX"], None]

    if 'Запись с наименованием схемы' in error_text:
        error_text_list = error_text.split()
        schema_name = error_text_list[4][1: -2]
        table_name = error_text_list[6][1: -1]
        field_name =  error_text_list[9][1: -1]

        return [d_aggr_errors["NOT_SETTINGS_TABLE"], (schema_name, table_name, field_name)]

    if 'attribute_name (' in error_text:
        error_text_list = error_text.split()
        attr = error_text_list[1][1:-1]
        return [d_aggr_errors["NOT_SETTINGS_ATTR"], attr]

    if 'invalid boundary specification' in error_text:
        return [d_aggr_errors["INVALID_STRUCTURE"], None]

    if 'task_group_code [] не добавлено в таблицу sys_task_group' in error_text:
        return [d_aggr_errors["NOT_TASK_GROUP_CODE"], None]
    
    #return ['неизвестное', None]
    return [d_aggr_errors["UNDEFINED"], None]
    

def read_errors_csv(path):
    rows = []
    with open(path, "r", encoding="utf-8-sig", newline="") as f:
        reader = csv.reader(f, delimiter=";")
        next(reader, None)  # header
        for row in reader:
            if len(row) < 2:
                continue
            file_path = row[0].strip()
            if file_path == "file_path":
                continue
            error_text = ";".join(row[1:]).strip()
            if file_path:
                rows.append((file_path, error_text))
    return rows


def main():
    
    rows = read_errors_csv(INPUT_CSV)
    print(f"Прочитано записей: {len(rows)}\n")

    groups = defaultdict(list)
    for file_path, error_text in rows:
        name, context = classify_error(error_text)
        was = ""
        to = ""
        if name == d_aggr_errors["PARTS_63"]:
            was = context
            to = "временное решение до принятие арх решения - брать название таблицы + хэш md5 от партиции и от сабпартиции и Greenplum самостоятельно обрежет до 63 символов или добавлять нумерацию для одинаковых имен партиций."
        
        if name == d_aggr_errors["NOT_SOURCE"]:
            was = f'нет объекта в модели {context}'
            to = 'на доработку аналитику?'
        
        if name == d_aggr_errors["NOT_FIELD"]:
            was = f'в таблице источнике TODO:[посмотреть какая таблица] отсутствует атрибут {context}'
            to = 'на доработку аналитику?'
        
        if name == d_aggr_errors["SYNTAX"]:
            was = 'TODO:[посмотреть из-за чего конкретно ошибка]'
            to = 'на доработку аналитику?'
        
        if name == d_aggr_errors["NOT_CTE"]:
            was = f'используется несуществуюшая CTE "{context}"'
            to = 'на доработку аналитику?'
        
        if name == d_aggr_errors["PARTS_DONT_GEN"]:
            was = f'no partitions specified at depth 1, не генерируются партиции для LIST типа партиций'
            to = 'нужна доработка выгрузки модели?'
        
        if name == d_aggr_errors["UNDEFINED"]:
            was = 'TODO[посмотреть руками и если что добавить в группу]'
            to = 'TODO[посмотреть руками и если что добавить в группу]'

        if name == d_aggr_errors["NOT_SETTINGS_TABLE"]:
            was = f'Таблицы {context[0]}.{context[1]} или только поля {context[2]} нет в системной таблицы system_core.sys_object_attrs'
            to = 'TODO[посмотреть почему нет таблицы в сис таблице]'

        if name == d_aggr_errors["INVALID_STRUCTURE"]:
            was = 'невалидное формирование партиций для объекта'
            to = 'нужна доработка выгрузки модели?'

        if name == d_aggr_errors["NOT_TASK_GROUP_CODE"]:
            was = 'нет task_group_code в мете'
            to = 'TODO[новая схема и ее нужно создать и добавить в мету?]'

        if name == d_aggr_errors["NOT_SETTINGS_ATTR"]:
            was = 'нет атрибута таблицы в мете'
            to = 'TODO[посмотреть почему нет атрибута?]'
            
        groups[name].append((str(datetime.datetime.now()).split()[0], file_path, name, was, to))

    for name in sorted(groups.keys()):
        items = groups[name]
        print(f"=== {name} ({len(items)}) ===")
    #    for dt, fl, nm_error, was, to in items:
    #        print(f"{dt}, {fl}, {nm_error}, {was}, {to}")


    # Пишем агрегированный CSV
    with open(OUT_CSV, "w", encoding="utf-8-sig", newline="") as f:
        writer = csv.writer(f, delimiter=";")
        #writer.writerow(["error_name", "file_path", "error_text"])
        for name in sorted(groups.keys()):
            for dt, fl, nm_error, was, to in groups[name]:
                object_name = fl.split("\\")[-1]
                if object_name == 'up.sql':
                     object_name = fl.split("\\")[6]
                writer.writerow([dt, object_name, nm_error,None, None, None, was, to])

            
# if __name__ == "__main__":
    #main()

In [46]:
main()


Прочитано записей: 275

=== не генерируются значения для списковых партиций (7) ===
=== невалидное формирование партиций (1) ===
=== неизвестное (1) ===
=== нет CTE (8) ===
=== нет атрибута в сис таблице system_core.sys_object_attrs (2) ===
=== нет атрибутов таблицы в сис таблице system_core.sys_object_attrs (15) ===
=== нет источника (155) ===
=== отсутсвует поле (25) ===
=== повторяющееся название таблицы партиции при превышение длины в 63 символа (25) ===
=== синтаксическая ошибка (8) ===
=== такой task_group_code нет в мете (28) ===
